# PART 04 · 5장 상관분석 로지스틱 회귀분석

교재 377~399쪽 본문 소스 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

#### 1. 컬럼의 역할

#### 2. 데이터 기본 정보

#### 3. 전체의 분석 흐름

#### 4. 공통 라이브러리와 데이터 불러오기

공통 라이브러리와 데이터를 불러오고 target 분포를 확인해둔다.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
path = (
    'data/part04_type3/logistic_regression_trial_signup/'
    'trial_signup.csv'
)
df = pd.read_csv(path)
print(df.shape)
print(df['signed_up'].value_counts().sort_index())
print(round(df['signed_up'].mean(), 4))

(520, 7)
signed_up
0    449
1     71
Name: count, dtype: int64
0.1365


## 01 로지스틱 회귀분석

## 02 로지스틱 회귀의 이해

### 1 선형 회귀와의 차이

#### 1. 선형 회귀와 로지스틱 회귀의 차이

### 2 확인 값과 식의 흐름

#### 1. 시험에서 확인하는 값

#### 2. 로지스틱 회귀 식의 흐름

## 03 오즈

### 1 확률과 오즈 계산

#### 1. 확률과 오즈의 계산식

#### 2. 전체 가입 확률과 오즈

전체 데이터의 가입 확률과 가입 오즈를 계산해본다.

In [2]:
success_count = int(df['signed_up'].sum())
total_count = len(df)
failure_count = total_count-success_count
probability = success_count/total_count
odds = success_count/failure_count
print('success: ', success_count)
print('failure: ', failure_count)
print('probability: ', round(probability, 4))
print('odds: ', round(odds, 4))

success:  71
failure:  449
probability:  0.1365
odds:  0.1581


### 2 오즈 변환과 활용

#### 1. 확률과 오즈의 변환

변환식을 코드로 확인해본다.

In [3]:
p = 0.1365
odds_from_p = p/(1-p)
p_from_odds = odds_from_p/(1+odds_from_p)
print(round(odds_from_p, 4))
print(round(p_from_odds, 4))

0.1581
0.1365


#### 2. 이메일 열람 여부별 가입 오즈

이메일 열람 여부에 따라 집단을 나눠 각 집단의 확률과 오즈를 계산해본다.

In [4]:
summary = df.groupby('email_opened')['signed_up'].agg(
    total='count',
    success='sum'
)
summary['failure'] = summary['total']-summary['success']
summary['probability'] = summary['success']/summary['total']
summary['odds'] = summary['success']/summary['failure']
print(summary.round(4))

              total  success  failure  probability    odds
email_opened                                              
0               226       19      207       0.0841  0.0918
1               294       52      242       0.1769  0.2149


## 04 오즈비

### 1 오즈비의 기준과 계산

#### 1. 오즈비의 기준

#### 2. 이메일 열람 집단의 원시 오즈비

두 집단의 오즈로 원시 오즈비를 직접 계산해본다.

In [5]:
odds_not_opened = 19 / 207
odds_opened = 52/242
raw_odds_ratio = odds_opened/odds_not_opened
print(format(odds_not_opened, '.4f'))
print(format(odds_opened, '.4f'))
print(format(raw_odds_ratio, '.4f'))

0.0918
0.2149
2.3410


### 2 회귀계수와 오즈비

#### 1. 회귀계수와 오즈비

로지스틱 회귀모형에서는 회귀계수에 지수함수 exp( )를 적용해 오즈비를 구한다. 회귀계수는 로그 오즈의 변화량이고, 오즈비는 이를 배수 형태로 바꾼 값이다.

In [6]:
coef = 0.8754809444634699  # 반올림 전 회귀계수
odds_ratio = np.exp(coef)
print(format(odds_ratio, '.4f'))

2.4000


#### 2. 여러 단위 증가의 오즈비

수치형 변수는 한 단위 증가뿐 아니라 여러 단위 증가에 대한 오즈비도 계산할 수 있다. visited_pages의 회귀계수가 0.1867일 때 페이지 수가 3개 증가하면 다음과 같다.

In [7]:
coef_pages = 0.18673885864998302  # 반올림 전 회귀계수
odds_ratio_3 = np.exp(coef_pages*3)
print(format(odds_ratio_3, '.4f'))

1.7511


## 05 로지스틱 회귀모형 생성

### 1 모형 생성

#### 1. 모형 생성 순서

#### 2. 기본 모형 적합

네 개의 독립변수로 기본 모형을 적합해본다.

In [8]:
feature_cols = [
    'age',
    'visited_pages',
    'email_opened',
    'previous_purchase'
]
X = sm.add_constant(df[feature_cols])
y = df['signed_up']
model = sm.Logit(y, X).fit(disp=False)
result_table = pd.DataFrame({
    'coef': model.params,
    'std_err': model.bse,
    'z': model.tvalues,
    'p_value': model.pvalues,
    'odds_ratio': np.exp(model.params)
})
print('x_shape: ', X.shape)
print('y_shape: ', y.shape)
print('converged: ', model.mle_retvals['converged'])
print('iterations: ', model.mle_retvals['iterations'])
print('pseudo_r_squared: ', round(model.prsquared, 4))
print(result_table.round(4))

x_shape:  (520, 5)
y_shape:  (520,)
converged:  True
iterations:  7
pseudo_r_squared:  0.0768
                     coef  std_err       z  p_value  odds_ratio
const             -4.9193   0.7057 -6.9704   0.0000      0.0073
age                0.0305   0.0119  2.5547   0.0106      1.0309
visited_pages      0.1867   0.0604  3.0903   0.0020      1.2053
email_opened       0.8755   0.2907  3.0114   0.0026      2.4000
previous_purchase  0.6645   0.2745  2.4206   0.0155      1.9435


### 2 결과 확인

#### 1. 결과 객체에서 필요한 값 꺼내기

#### 2. 이진 범주의 기준 상태

## 06 회귀계수 해석

### 1 계수의 부호와 오즈비

#### 1. 계수의 부호와 오즈비

### 2 변수별 계수 해석

#### 1. 수치형 변수의 계수

#### 2. 0·1 변수의 계수

#### 3. 로그 오즈와 예측 확률 변화

첫 번째 방문자의 다른 조건을 그대로 두고 visited_pages만 4개에서 5개로 바꾸면 로그 오즈와 예측 확률은 다음처럼 변한다.

In [9]:
row = X.iloc[[0]].copy()
row_4 = row.copy()
row_4['visited_pages'] = 4
row_5 = row.copy()
row_5['visited_pages'] = 5
log_odds_4 = float(row_4.iloc[0]@model.params)
log_odds_5 = float(row_5.iloc[0]@model.params)
prob_4 = float(model.predict(row_4).iloc[0])
prob_5 = float(model.predict(row_5).iloc[0])
print(format(log_odds_4, '.4f'), format(prob_4, '.4f'))
print(format(log_odds_5, '.4f'), format(prob_5, '.4f'))

-3.1971 0.0393
-3.0103 0.0470


#### 4. 기본 모형의 회귀계수

## 07 오즈비 해석

### 1 기본 모형의 오즈비

#### 1. 기본 모형의 오즈비

회귀계수에 지수함수를 적용하면 변수별 오즈비가 된다.

In [10]:
odds_ratio = np.exp(model.params)
print(odds_ratio.round(4))

const                0.0073
age                  1.0309
visited_pages        1.2053
email_opened         2.4000
previous_purchase    1.9435
dtype: float64


### 2 변수별 오즈비 해석

#### 1. 수치형 변수의 오즈비

#### 2. 0과 1 변수의 오즈비

#### 3. 변수별 오즈비 정리

## 08 p-value 해석

### 1 변수 유의성 확인

#### 1. 변수별 귀무가설

#### 2. 기본 모형의 유의성

### 2 확장 모형과 결론

#### 1. region을 포함한 확장 모형

범주형 변수 region을 더미 변수로 바꿔 포함한 확장 모형을 적합해본다.

In [11]:
region_dummy = pd.get_dummies(
    df[['region']],
    drop_first=True,
    dtype=int
)
X_ext = pd.concat(
    [df[feature_cols], region_dummy],
    axis=1
)
X_ext = sm.add_constant(X_ext)
model_ext = sm.Logit(y, X_ext).fit(disp=False)
region_result = pd.DataFrame({
    'coef': model_ext.params,
    'p_value': model_ext.pvalues,
    'odds_ratio': np.exp(model_ext.params)
}).loc[['region_local_city', 'region_suburban']]
print(region_result.round(4))

                     coef  p_value  odds_ratio
region_local_city  0.1745   0.5414      1.1907
region_suburban   -1.0055   0.0234      0.3658


#### 2. 통계적으로 안전한 결론 문장

#### 3. 유의한 변수 선택과 모형 재적합

로지스틱 회귀에서도 문제에서 p-value 기준으로 변수를 선택하라고 하면 절편을 제외한 변수별 p-value에서 컬럼 이름을 추출한다. 범주형 변수는 더미 변환 후 생성된 컬럼 이름을 기준으로 선택한다.

In [12]:
features = [
    'age',
    'visited_pages',
    'email_opened',
    'previous_purchase',
    'region'
]
X_full = pd.get_dummies(
    df[features],
    columns=['region'],
    drop_first=True,
    dtype=int
)
X_full = sm.add_constant(
    X_full,
    has_constant='add'
)
y = df['signed_up']
full_model = sm.Logit(
    y, X_full
).fit(disp=False)
feature_pvalues = full_model.pvalues.drop('const')
sig_vars = feature_pvalues[
    feature_pvalues < 0.05
].index.tolist()
X_refit = sm.add_constant(
    X_full[sig_vars],
    has_constant='add'
)
refit_model = sm.Logit(
    y, X_refit
).fit(disp=False)
print('selected_variables: ', sig_vars)
print(refit_model.params.round(4))

selected_variables:  ['age', 'visited_pages', 'email_opened', 'previous_purchase', 'region_suburban']
const               -4.8191
age                  0.0304
visited_pages        0.1947
email_opened         0.9029
previous_purchase    0.6928
region_suburban     -1.0825
dtype: float64


문제에서 요구하는 값은 다음과 같이 필요한 식만 골라 계산한다.

In [13]:
print('count: ', len(sig_vars))
print(
    'coef_sum_with_const: ',
    round(refit_model.params.sum(), 4)
)
print(
    'age_5_or: ',
    round(np.exp(refit_model.params['age'] * 5), 4)
)
pred_prob = refit_model.predict(X_refit)
print(
    'prob_over_03: ',
    int((pred_prob > 0.3).sum())
)

count:  5
coef_sum_with_const:  -4.0808
age_5_or:  1.1641
prob_over_03:  42


## 09 예측 결과 해석

### 1 예측 확률 계산

#### 1. 예측 확률과 예측 분류

#### 2. 예측 확률 계산

모형의 예측 확률을 계산하고 기준값 0.5로 예측 분류까지 만들어 본다.

In [14]:
pred_prob = model.predict(X)
pred_class = (pred_prob >= 0.5).astype(int)
pred_table = df[[
    'visitor_id',
    'signed_up'
]].copy()
pred_table['pred_prob'] = pred_prob
pred_table['pred_class'] = pred_class
print(pred_table.head().round(4))
print('min_prob: ', round(pred_prob.min(), 4))
print('max_prob: ', round(pred_prob.max(), 4))
print('mean_prob: ', round(pred_prob.mean(), 4))
print(pred_table['pred_class'].value_counts().sort_index())

  visitor_id  signed_up  pred_prob  pred_class
0      V0001          0     0.0393           0
1      V0002          0     0.0734           0
2      V0003          0     0.1179           0
3      V0004          0     0.0811           0
4      V0005          0     0.0807           0
min_prob:  0.0216
max_prob:  0.5561
mean_prob:  0.1365
pred_class
0    518
1      2
Name: count, dtype: int64


### 2 예측 결과 활용

#### 1. 예측 확률이 가장 높은 행

예측 확률이 가장 높은 행을 정렬로 확인해본다.

In [15]:
top_prob = pred_table.sort_values(
    'pred_prob',
    ascending=False
).head(5)
print(top_prob.round(4))

    visitor_id  signed_up  pred_prob  pred_class
494      V0495          1     0.5561           1
368      V0369          1     0.5106           1
120      V0121          0     0.4413           0
189      V0190          0     0.4218           0
69       V0070          1     0.4097           0


#### 2. 기준값의 의미

#### 3. 선별된 변수 구조로 새로운 값 예측

새 관측치의 설계행렬은 재적합에 사용한 X_refit.columns에 맞춘다. 선택되지 않은 후보 변수는 reindex( ) 과정에서 제외된다.

In [16]:
new_data = pd.DataFrame({
    'age': [35],
    'visited_pages': [6],
    'email_opened': [1],
    'previous_purchase': [0],
    'region_local_city': [0],
    'region_suburban': [1]
})
new_X = sm.add_constant(
    new_data,
    has_constant='add'
)
new_X = new_X.reindex(
    columns=X_refit.columns,
    fill_value=0
)
new_prob = refit_model.predict(new_X)
print(round(new_prob.iloc[0], 4))

0.0591


## 10 오분류율

### 1 오분류율 계산

#### 1. 오분류 개수와 오분류율

예측 분류와 실젯값을 비교해 오분류 개수와 오분류율을 계산해본다.

In [17]:
wrong = pred_class != y
error_count = int(wrong.sum())
error_rate = float(wrong.mean())
accuracy = float((pred_class == y).mean())
print('error_count: ', error_count)
print('error_rate: ', round(error_rate, 4))
print('accuracy: ', round(accuracy, 4))

error_count:  69
error_rate:  0.1327
accuracy:  0.8673


#### 2. 실젯값과 예측값의 교차표

실젯값과 예측값의 교차표를 만들어 본다.

In [18]:
confusion = pd.crosstab(
    y,
    pred_class,
    rownames=['actual'],
    colnames=['pred']
)
print(confusion)

pred      0  1
actual        
0       449  0
1        69  2


### 2 정확도와의 관계와 주의점

#### 1. 정확도와 오분류율의 관계

#### 2. 해석할 때 주의할 점